In [50]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}
url = "https://www.worldometers.info/coronavirus/"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml") 
table = soup.find("table", id="main_table_countries_today")

- Đầu tiên ta sẽ cài header phù hợp để đăng nhập vào web 
- Sau khi gửi respone ta sẽ lọc ra bảng dữ liệu mà ta cần có id là main_table_countries_today

In [51]:
data = []
rows = table.find_all("tr")
for row in rows:
    cells = row.find_all(["td", "th"])
    row_data = [cell.get_text(strip=True) for cell in cells]
    if row_data:
        data.append(row_data)
df = pd.DataFrame(data[1:], columns=data[0])
df = df[df['#'] != ''].reset_index(drop=True)

- Đầu tiên ta tạo 1 df rỗng 
- Duyệt lấy tất cả hàng trong bảng và lọc ra những hàng có header và dữ liệu, sau đó tiền xử lí nhẹ bằng cách xóa khoảng trắng
- Drop các hàng có cột '#' chứa giá trị trống do lỗi cào

In [ ]:
# Đổi tên cho dễ làm việc :v
df = df.rename(columns={
    "#": "Rank",
    "Country,Other": "Country",
    "Total Cases": "TotalCases",
    "New Cases": "NewCases",
    "Total Recovered": "TotalRecovered"
})

In [53]:
display(df.head(20))

,Rank,Country,TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,"Serious,Critical",...,TotalTests,Tests/1M pop,Population,Continent,1 Caseevery X ppl,1 Deathevery X ppl,1 Testevery X ppl,New Cases/1M pop,New Deaths/1M pop,Active Cases/1M pop
0,1,USA,"111,820,082",,"1,219,487",,"109,814,428",,"786,167",940,...,"1,186,851,502","3,544,901","334,805,269",North America,3,275,0,,,"2,348"
1,2,India,"45,035,393",,"533,570",,N/A,N/A,N/A,N/A,...,"935,879,495","665,334","1,406,631,776",Asia,31,"2,636",2,,,0.4
2,3,France,"40,138,560",,"167,642",,"39,970,918",,0,,...,"271,490,188","4,139,547","65,584,518",Europe,2,391,0,,,
3,4,Germany,"38,828,995",,"183,027",,"38,240,600",,"405,368",N/A,...,"122,332,384","1,458,359","83,883,596",Europe,2,458,1,,,"4,833"
4,5,Brazil,"38,743,918",,"711,380",,"36,249,161",,"1,783,377",N/A,...,"63,776,166","296,146","215,353,593",South America,6,303,3,,,"8,281"
5,6,S. Korea,"34,571,873",,"35,934",,"34,535,939",,0,,...,"15,804,065","307,892","51,329,899",Asia,1,"1,428",3,,,
6,7,Japan,"33,803,572",,"74,694",,N/A,N/A,N/A,N/A,...,"100,414,883","799,578","125,584,838",Asia,4,"1,681",1,,,"95,582"
7,8,Italy,"26,723,249",,"196,487",,"26,361,218",,"165,544",22,...,"281,126,449","4,665,010","60,262,770",Europe,2,307,0,,,"2,747"
8,9,UK,"24,910,387",,"232,112",,"24,678,275",,0,N/A,...,"522,526,476","7,628,357","68,497,907",Europe,3,295,0,,,
9,10,Russia,"24,124,215",,"402,756",,"23,545,818",,"175,641",N/A,...,"273,400,000","1,875,095","145,805,947",Europe,6,362,1,,,"1,205"


In [54]:
df = df[["Country", "TotalCases", "NewCases", "TotalRecovered"]]

In [55]:
for col in df.columns[df.columns != "Country"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace("+", "")
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.fillna(0)

- Tiền xử lí các giá trị số do lúc cào, những dữ liệu này đang ở dạng chuỗi vậy nên ta cần đổi sang kiểu numeric đồng thời fill các cột chứa giá trị N/A thành 0

In [56]:
display(df.head(20))

,Country,TotalCases,NewCases,TotalRecovered
0,USA,111820082,0.0,109814428.0
1,India,45035393,0.0,0.0
2,France,40138560,0.0,39970918.0
3,Germany,38828995,0.0,38240600.0
4,Brazil,38743918,0.0,36249161.0
5,S. Korea,34571873,0.0,34535939.0
6,Japan,33803572,0.0,0.0
7,Italy,26723249,0.0,26361218.0
8,UK,24910387,0.0,24678275.0
9,Russia,24124215,0.0,23545818.0


### a) Tìm 5 quốc gia có số ca nhiễm (Total case) nhiều nhất.

In [62]:
display(df.sort_values(by="TotalCases", ascending=False)["Country"].head(5))

0        USA
1      India
2     France
3    Germany
4     Brazil
Name: Country, dtype: object

- Ta chỉ cần sort theo giá trị TotalCases giảm dần sau đó in ra 5 quốc gia đầu tiên

### b)Quốc gia nào có số ca nhiễm mới cao nhất?

In [63]:
MaxNewCases = df["NewCases"].max()
display(df[df["NewCases"] == MaxNewCases]["Country"])

0                 USA
1               India
2              France
3             Germany
4              Brazil
            ...      
226           Tokelau
227      Vatican City
228    Western Sahara
229        MS Zaandam
230             China
Name: Country, Length: 231, dtype: object

- Do có thể có nhiều quốc gia cùng đứng hạng nhất về số ca nhiễm, ta tìm giá trị lớn nhất của số ca nhiễm mới, sau đó lọc ra những hàng có số ca nhiễm mới bằng số ca nhiễm mới cao nhất

### c) Tính tỉ lệ tổng số ca bình phục trên tổng số ca nhiễm. Xác định 3 quốc gia có tỉ lệ bình phục cao nhất. 

In [ ]:
df['RecoveredRate'] = df["TotalRecovered"]/df["TotalCases"] 
MaxRecoveredRate = df["RecoveredRate"].max()
display(df[df["RecoveredRate"] == MaxRecoveredRate]["Country"])

222    Falkland Islands
227        Vatican City
Name: Country, dtype: object

- Đầu tiên ta tạo ra 1 cột mới là tỉ lệ hồi phục, sau đó làm tương tự câu b